# 02 · Capa bronce: CSV → tabla Delta

**Qué hace:** lee los CSV de cada año y los escribe en la tabla Delta `bronze_contratos`
(carpeta `datos/delta/bronze_contratos/`), particionada por `anio_carga`.

**Reglas de bronce:**
1. **Todo como texto (string).** Bronce es una copia fiel; si convirtiéramos tipos aquí y
   la conversión fallara, perderíamos el dato original. Los tipos se corrigen en plata.
2. **Sin datos personales.** Las columnas personales se descartan antes de escribir.
3. **Recarga por año con `replaceWhere`.** Volver a cargar 2019 reemplaza SOLO la partición
   `anio_carga = 2019`; los demás años no se tocan. Así el notebook se puede repetir sin
   duplicar filas (es *idempotente*).

**¿Por qué Delta y no CSV/Parquet?** Delta agrega un registro de transacciones
(`_delta_log`): escrituras atómicas (todo o nada), esquema validado e historial de
versiones (*time travel*).

## Configuración: importar `src/secop` y crear la sesión de Spark

1. Buscamos la carpeta del repositorio (la que contiene `src/secop`) subiendo desde la
   carpeta del notebook y añadimos `src` a `sys.path`, la lista de carpetas donde Python
   busca módulos. Así `import secop` funciona sin instalar nada.
2. `entorno.crear_spark()` crea una sesión de **Spark en modo local** con Delta Lake: el
   mismo motor que en un clúster, usando los núcleos del Codespace.

Si cambias código en `src/`, reinicia el kernel (botón *Restart*) para que Python lo
vuelva a cargar.

In [ ]:
import sys
from pathlib import Path


def buscar_raiz_repo() -> Path:
    """Devuelve la carpeta raíz del repo (la que contiene src/secop)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "src" / "secop").is_dir():
            return carpeta
    raise RuntimeError("No encontré src/secop: abre el notebook desde la carpeta del repositorio.")


RAIZ_REPO = buscar_raiz_repo()
if str(RAIZ_REPO / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ_REPO / "src"))

import secop
from secop import config, entorno

spark = entorno.crear_spark()
print("secop", secop.__version__, "| Spark", spark.version, "| datos en", config.raiz_datos())

## 1. Parámetros
- `ANIOS`: años a cargar.
- `RECREAR`: `si` borra la tabla (y su historial) y la crea de nuevo. Úsalo solo si la
  tabla existe con otro esquema o partición.

In [ ]:
ANIOS = [int(a) for a in entorno.parametro("ANIOS", "2019,2020").split(",") if a.strip()]
RECREAR = entorno.parametro("RECREAR", "no") == "si"
if RECREAR:
    entorno.borrar_tabla(config.TABLA_BRONCE)
    print("Tabla eliminada; se creará de nuevo.")
print("Años:", ANIOS)

## 2. Cargar cada año

- Antes de leer, se verifica que todos los CSV del año tengan el **mismo encabezado**:
  Spark asigna columnas por posición usando el primer archivo, y una mezcla de formatos
  desalinearía los datos sin dar error.
- `_metadata.file_name` es una columna oculta que Spark ofrece al leer archivos:
  guardamos de qué archivo viene cada fila (trazabilidad).
- Leemos la carpeta del año directamente (`.../anio=2019`), no la carpeta padre. Si se
  leyera `.../contratos`, Spark descubriría `anio=` como una columna de partición
  llamada `anio`, distinta de nuestra `anio_carga`.

In [ ]:
import os

from pyspark.sql import functions as F

from secop.bronce import preparar_bronce
from secop.validacion import encabezados_distintos, exigir_sin_personales


def leer_csv_anio(anio: int):
    carpeta = config.ruta_anio(anio)
    if not os.path.isdir(carpeta):
        raise FileNotFoundError(f"No existe {carpeta}. Ejecuta primero el notebook 01.")
    rutas = sorted(os.path.join(carpeta, f) for f in os.listdir(carpeta) if f.endswith(".csv"))
    if not rutas:
        raise FileNotFoundError(f"No hay CSV en {carpeta}. Ejecuta primero el notebook 01.")
    grupos = encabezados_distintos(rutas)
    if len(grupos) > 1:
        raise ValueError(
            f"{anio}: los CSV tienen encabezados distintos. "
            "Vuelve a descargar el año completo con un solo método."
        )
    return (
        spark.read.option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .option("pathGlobFilter", "*.csv")
        .csv(carpeta)
        .withColumn("archivo_origen", F.col("_metadata.file_name"))
    )


for anio in ANIOS:
    bronce = preparar_bronce(leer_csv_anio(anio), anio)
    exigir_sin_personales(bronce.columns)  # doble verificación antes de escribir
    entorno.escribir_tabla(
        bronce,
        config.TABLA_BRONCE,
        particion="anio_carga",  # se usa al crear la tabla
        reemplazar_donde=f"anio_carga = {anio}",  # se usa si la tabla ya existe
    )
    print(f"{anio}: partición escrita")

## 3. Verificación
Filas por año (referencia de la descarga previa: 2019 → 142.090; 2020 → 356.623) y
confirmación de que la tabla no tiene columnas personales.

In [ ]:
tabla = entorno.leer_tabla(spark, config.TABLA_BRONCE)
exigir_sin_personales(tabla.columns)
print("Columnas en bronce:", len(tabla.columns))
entorno.mostrar(tabla.groupBy("anio_carga").count().orderBy("anio_carga"))

## 4. Historial de Delta (*time travel*)
Cada escritura crea una versión. Se puede leer una versión anterior con
`entorno.leer_version(spark, "bronze_contratos", n)`: útil para auditar o recuperar una
carga mala. En la carpeta de la tabla, cada versión es un archivo en `_delta_log/`.

In [ ]:
entorno.mostrar(
    entorno.historial(spark, config.TABLA_BRONCE).select(
        "version", "timestamp", "operation", "operationParameters"
    )
)

In [ ]:
# Ejemplo de time travel: cuántas filas tenía la versión 0 (la primera carga).
print("Filas en la versión 0:", entorno.leer_version(spark, config.TABLA_BRONCE, 0).count())